# Phase 0: Hardware Exploration
**Step 1: Loading the Bitstream and Understanding the HWH File**

In traditional FPGA development, software engineers have to manually hardcode physical memory addresses to talk to the hardware (e.g., `0x41200000`). 

PYNQ changes this entirely by using the **`.hwh` (Hardware Handoff) file**. When the hardware team compiles the Vivado project, Vivado generates this XML-based file. It contains the complete schematic of the FPGA design, including all IP blocks, their memory addresses, interrupts, and clock frequencies.

When we load the `.bit` file using `pynq.Overlay`, PYNQ secretly looks for a `.hwh` file with the exact same name, parses it, and dynamically creates a Python dictionary mapping the entire hardware architecture. Let's load the hardware into the Programmable Logic (PL).

In [2]:
from pynq import Overlay

# This command flashes the FPGA Programmable Logic (PL) with the bitstream
# AND parses the balancin.hwh file to build the system architecture in Python.
ol = Overlay("balancin.bit")

print("Bitstream loaded and HWH parsed successfully!")

Bitstream loaded and HWH parsed successfully!


To verify that the PL (Programmable Logic) was successfully programmed, we can query the overlay's status.

In [3]:
# Verify the bitstream is currently active on the FPGA
is_flashed = ol.is_loaded()
print(f"Is the FPGA programmed with our design? {is_flashed}")

Is the FPGA programmed with our design? True


**Step 2: Interrogating the IP Dictionary**

Because PYNQ parsed the `.hwh` file, it constructed a master dictionary of every intellectual property (IP) core present in the Programmable Logic. This dictionary is accessed via `ol.ip_dict`.

Instead of relying on a hardware schematic (like a Vivado PDF), we can dynamically ask the FPGA what hardware blocks are available to us as software engineers. First, let's look at the raw names of the IP blocks.

In [7]:
# List the names of all the hardware blocks discovered in the FPGA
ip_names = list(ol.ip_dict.keys())
print("Discovered IP Blocks:")
for name in ip_names:
    print(f"- {name}")

Discovered IP Blocks:
- axi_gpio_0
- axi_iic_0
- processing_system7_0


You should see names like `axi_gpio_0` (which we suspect is the motor) and `axi_iic_0` (which we suspect is the MPU6050 sensor). 

However, just knowing the names isn't enough. To communicate with these blocks, the CPU needs to know their **Physical Memory Address**. In a Zynq chip, the hardware is memory-mapped. This means that writing a number to a specific physical memory address doesn't save it to RAM; instead, it sends an electrical signal directly to the hardware pins!

Let's write a loop to extract the Name, Physical Address, and the Xilinx IP Type for everything in our system.

In [9]:
# Create a nicely formatted table of our hardware map
print(f"{'IP Block Name':<20} | {'Physical Address':<18} | {'Xilinx IP Type'}")
print("-" * 75)

for name, info in ol.ip_dict.items():
    # Safely extract physical address using .get() to avoid KeyError on IPs without memory mapping
    addr = info.get('phys_addr')
    phys_addr = hex(addr) if addr is not None else "N/A"
    
    # Safely extract the standard Xilinx type identifier
    ip_type = info.get('type', 'N/A')
    
    print(f"{name:<20} | {phys_addr:<18} | {ip_type}")

IP Block Name        | Physical Address   | Xilinx IP Type
---------------------------------------------------------------------------
axi_gpio_0           | 0x41200000         | xilinx.com:ip:axi_gpio:2.0
axi_iic_0            | 0x41600000         | xilinx.com:ip:axi_iic:2.1
processing_system7_0 | N/A                | xilinx.com:ip:processing_system7:5.5


**Step 3: Inspecting the Register Map (AXI GPIO)**

We know the `axi_gpio_0` block lives at the physical address `0x41200000`. But what exactly is at that address? 

Hardware blocks use **Registers** (small memory slots) to receive commands and data. For standard Xilinx IPs, PYNQ parses the `.hwh` file to figure out exactly how these registers are organized. 

Instead of guessing or reading a 100-page Xilinx datasheet, we can ask PYNQ to generate an interactive map of the GPIO's internal registers. Let's inspect our motor controller.

In [10]:
# Access the specific IP block
gpio_ip = ol.axi_gpio_0

# Display the interactive register map
# In Jupyter, this will render as a nice readable table
gpio_ip.register_map

RegisterMap {
  GPIO_DATA = Register(CH1_DATA=0),
  GPIO_TRI = Register(CH1_TRI=1023),
  GPIO2_DATA = Register(CH2_DATA=0),
  GPIO2_TRI = Register(CH2_TRI=4294967295),
  GIER = Register(INT_EN=1),
  IP_ISR = Register(CH1_INT_S=0, CH2_INT_S=0),
  IP_IER = Register(CH1_INT_EN=0, CH2_INT_EN=0)
}

When you run the cell above, you should see a table of registers. Pay attention to two specific registers:
1. **`Channel_1_DATA`** (Usually at offset `0x00`): This is where the actual value (0 to 1023 for your PWM) is stored. 
2. **`Channel_1_TRI`** (Usually at offset `0x04`): This is the "Tri-state" or direction register. It tells the hardware if the pins are Inputs (1) or Outputs (0).

To send a PWM signal to the motor, we need to make sure the direction is set to Output, and then write a number into the DATA register. PYNQ's `register_map` allows us to interact with these registers by name!

In [13]:
# Let's read the current state of the Channel 1 Data Register
current_data = gpio_ip.register_map.GPIO_DATA
print(f"Current value in GPIO_DATA register: {current_data}")

# Let's read the Tri-state (Direction) register
current_dir = gpio_ip.register_map.GPIO_TRI
print(f"Current value in GPIO_TRI (Direction) register: {hex(current_dir)}")

Current value in GPIO_DATA register: 0x0
Current value in GPIO_TRI (Direction) register: 0x3ff


**Step 4: Raw Memory-Mapped I/O (MMIO) Control**

Every "High-Level" driver in PYNQ (like `AxiGPIO`) is actually just a wrapper around a lower-level class called **`MMIO`**. 

As a software engineer, you can think of the FPGA as a block of RAM. 
- The **Base Address** (which we found in Step 2: `0x41200000`) is the start of the hardware block.
- The **Offsets** are the specific "slots" within that block.

For the AXI GPIO:
- Offset `0x00`: The Data Register (`GPIO_DATA`)
- Offset `0x04`: The Tri-state Register (`GPIO_TRI`)

Let's use the `MMIO` class to physically configure the pins as outputs and send a pulse to the motor.

In [14]:
from pynq import MMIO

# Define the physical address we discovered earlier
GPIO_BASE_ADDR = 0x41200000
ADDRESS_RANGE = 0x10  # We only need 16 bytes of space

# Initialize the MMIO object
# This maps the physical hardware memory into our Python process
motor_mmio = MMIO(GPIO_BASE_ADDR, ADDRESS_RANGE)

print("MMIO Link established with the hardware.")

MMIO Link established with the hardware.


Now, let's perform the "Bare Metal" sequence to spin the motor:
1. **Set Direction:** Write `0` to offset `0x04`. In GPIO, a `0` means "Output" and a `1` means "Input".
2. **Write Value:** Write a number between `0` and `1023` to offset `0x00`.

*Safety Note: We will write a small value first (200) to ensure the motor doesn't jump too hard.*

In [17]:
# 1. Set the Tri-state register to 0 (All pins as Output)
motor_mmio.write(0x04, 0x0)

# 2. Write a PWM value to the Data register (Offset 0x00)
# Let's write 500 (roughly 50% power if max is 1023)
motor_mmio.write(0x00, 500)

print("Value 500 written to physical memory. Check your motor!")

Value 500 written to physical memory. Check your motor!


In [20]:
# To turn the motor off, write 0 to the data offset
motor_mmio.write(0x00, 1023)
print("Motor stopped.")

Motor stopped.


**Step 5: Discovering the Sensor Interface (AXI IIC)**

The motor is controlled by a simple GPIO (General Purpose Input/Output), but the **MPU6050 IMU** (Inertial Measurement Unit) is more complex. It communicates via the **I2C (Inter-Integrated Circuit) protocol**.

In our hardware discovery (Step 2), we found a block called `axi_iic_0` at address `0x41600000`. Unlike the GPIO, which just holds a number, this I2C controller has a "FIFO" (First-In-First-Out) buffer. To talk to the sensor, we have to:
1. Write the sensor's address (`0x68`) to the I2C controller.
2. Write the register we want to read (like the Accelerometer Z-axis) into the controller's transmit buffer.
3. Tell the controller to start the electrical transaction on the physical wires.

Let's look at the register map of this controller to see how it differs from the GPIO.

In [21]:
# Access the I2C IP block
i2c_ip = ol.axi_iic_0

# Display the register map
i2c_ip.register_map

RegisterMap {
  GIE = Register(GIE=0),
  ISR = Register(int0=0, int1=0, int2=0, int3=0, int4=1, int5=0, int6=1, int7=1),
  IER = Register(int0=0, int1=0, int2=0, int3=0, int4=0, int5=0, int6=0, int7=0),
  SOFTR = Register(RKEY=write-only),
  CR = Register(EN=0, TX_FIFO_Reset=0, MSMS=0, TX=0, TXAK=0, RSTA=0, GC_EN=0),
  SR = Register(ABGC=0, AAS=0, BB=0, SRW=0, TX_FIFO_Full=0, RX_FIFO_Full=0, RX_FIFO_Empty=1, TX_FIFO_Empty=1),
  TX_FIFO = Register(D7_D0=write-only, Start=write-only, Stop=write-only),
  RX_FIFO = Register(D7_D0=0),
  ADR = Register(Slave_Address=0),
  TX_FIFO_OCY = Register(Occupancy_Value=0),
  RX_FIFO_OCY = Register(Occupancy_Value=0),
  TEN_ADR = Register(MSB_of_Slave_Address=0),
  RX_FIFO_PIRQ = Register(Compare_Value=0),
  GPO = Register(General_Purpose_Outputs=0),
  TSUSTA = Register(TSUSTA=570),
  TSUSTO = Register(TSUSTO=500),
  THDSTA = Register(THDSTA=430),
  TSUDAT = Register(TSUDAT=55),
  TBUF = Register(TBUF=500),
  THIGH = Register(THIGH=493),
  TLOW = Regi

Looking at the `register_map` for `axi_iic_0`, you will see much more complex registers:
* **`DGIER` / `IPIER`**: Interrupt controllers (used for high-speed sensor timing).
* **`TX_FIFO`**: Where we "drop" data we want to send to the MPU6050.
* **`RX_FIFO`**: Where the data coming back from the sensor is stored.
* **`CR` (Control Register)**: The "Command Center" where we tell the hardware to START or STOP the I2C transaction.

**Conclusion of Phase 0:**
We have successfully:
1. Verified the hardware is alive.
2. Discovered the memory addresses (`0x41200000` and `0x41600000`).
3. Verified the register structure.
4. Discovered the **Inverse Logic** of the motor.

We are now ready to leave the "Exploration" phase and move to **Phase 1: Encapsulation**, where we will write professional Python classes so we never have to think about these raw memory addresses again.